# Choose the decoder setting

Picks `emission_scale` and `min_possession_s` on external data only, from the saved model's probabilities, without
training anything. It writes `decoder_choice.json` (read by `run_decoder.ipynb`) and `decoder_tuning.json` (every
setting compared).

The candidates are the grid of `tune_decoder` (`helper_decoder.ipynb`), 7 values of `emission_scale` times 3 of
`min_possession_s`. Each gets its mean spellF1 over the two external validation matches, J03WR9 and METRICA3, labelled
by the model trained without them (`train_model.ipynb`). Ties go to the better macroF1, then to the longer minimum.

In [ ]:
import os
from pathlib import Path

PIPELINE_DIR = Path.cwd().resolve()     # notebooks run from possession/
assert (PIPELINE_DIR / "run_decoder.ipynb").is_file(), "run this notebook from the possession/ folder"
REPO_ROOT = PIPELINE_DIR.parent
import json
import sys

import numpy as np

NAME = "possession_v1"
CHOICE = PIPELINE_DIR / "decoder_choice.json"
REPORT = PIPELINE_DIR / "decoder_tuning.json"


def notebook_namespace():
    """The definitions and constants of run_decoder.ipynb."""
    sys.path.insert(0, str(PIPELINE_DIR / "_vendor"))        # kloppy, if installed apart (pip --target possession/_vendor)
    os.environ.setdefault("MPLBACKEND", "Agg")
    nb = json.loads((PIPELINE_DIR / "run_decoder.ipynb").read_text(encoding="utf-8"))
    ns = {"__name__": "__main__"}
    for i, cell in enumerate(nb["cells"]):
        src = "".join(cell["source"])
        if cell["cell_type"] == "code" and ("load_notebook_defs" in src or "EMISSION_SCALE =" in src):
            exec(compile(src, "run_decoder.ipynb::cell{}".format(i), "exec"), ns)
    return ns


def choose():
    ns = notebook_namespace()
    bundle = ns["load_bundle"](NAME)
    val_ids = list(bundle["meta"]["info"]["val_matches"])
    val = np.load(PIPELINE_DIR / "artifacts" / (NAME + "_val_probs.npz"))
    val_probs = [val["heldout_probs_" + m] for m in val_ids]
    val_labels = [ns["ids_to_labels"](val["y_" + m]) for m in val_ids]
    val_blocks = [[tuple(int(x) for x in b) for b in val["blocks_" + m]] for m in val_ids]

    _, table = ns["tune_decoder"](val_probs, val_labels, val_blocks, bundle["dec_cfg"])   # sorted best first
    rows = [{"emission_scale": float(r["emission_scale"]), "min_possession_s": float(r["min_possession_s"]),
             "spellF1": float(r["spellF1"]), "macroF1": float(r["macroF1"])} for r in table.to_dict("records")]
    chosen = rows[0]
    for r in rows[:6]:
        print(json.dumps(r))
    print("chosen", json.dumps(chosen))
    with open(CHOICE, "w") as handle:
        json.dump({"emission_scale": chosen["emission_scale"], "min_possession_s": chosen["min_possession_s"],
                   "rule": "best mean spellF1 over the external validation matches J03WR9 and METRICA3 "
                           "(possession/choose_decoder.ipynb)"}, handle, indent=1)
        handle.write("\n")
    with open(REPORT, "w") as handle:
        json.dump({"val_matches": val_ids, "chosen": chosen, "rows": rows}, handle, indent=1)
        handle.write("\n")

In [ ]:
choose()